# 🎯 CPZÃO · Notebook 02 — Treino, Métricas e Export ONNX

**Auditoria de Acessibilidade em Calçadas** · YOLOv8n → ONNX → PWA mobile

Treina o detector, analisa as métricas exigidas pela rubrica (mAP@50, Precision, Recall,
matriz de confusão) e exporta o modelo no formato que o app mobile consome.

### Resolução de entrada

Este notebook treina em **`imgsz=192`**, que é o que o modelo entregue usa.

O app **detecta o tamanho de entrada sozinho** (sonda 192/256/320/416/640 com uma inferência
dummy ao carregar o modelo), então esse valor não é uma restrição imposta pelo código do app —
mudar `IMGSZ` aqui basta, sem tocar em uma linha do `app/`.

> 📉 A seção **Análise crítica**, no fim deste notebook, documenta por que 192 limita as
> classes finas e o que mudaria numa segunda iteração. Vale a leitura: é a parte analítica
> do trabalho.

### Pré-requisito

Nenhum. Se o dataset não existir neste runtime, a célula de configuração o remonta sozinha.

> 🖥️ **Runtime:** `Ambiente de execução → Alterar tipo de ambiente de execução → GPU (T4)`

## 1. Ambiente

In [ ]:
!pip install -q ultralytics onnx onnxruntime onnxslim

import os, sys, json, glob, shutil, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
from ultralytics import YOLO

print("torch       :", torch.__version__)
import ultralytics; print("ultralytics :", ultralytics.__version__)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("device      :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU ⚠️ (lento — ative a GPU)")

## 2. Configuração

Não precisa preencher nada.

> ⚠️ **No Colab cada notebook abre um runtime separado.** O dataset que o notebook 01 montou
> em `/content/` **não existe** nesta sessão. A célula abaixo detecta isso e remonta o dataset
> sozinha (~6 min), direto do Zenodo. O resultado é idêntico ao do notebook 01 — a seleção é
> determinística, com seed fixa.
>
> Se você rodou o notebook 01 **neste mesmo runtime**, ela só confirma e segue.

In [ ]:
# ─── Configuração ───
CLASSES  = ["degrau", "escada", "corrimao"]
IMGSZ    = 192                              # o app detecta o tamanho do modelo automaticamente
ALVO     = 600                              # usado só se precisar remontar o dataset
DEST     = Path("/content/dataset_calcadas")
DATA_YAML = str(DEST / "data.yaml")

# No Colab, cada notebook abre um runtime SEPARADO: o dataset que o notebook 01
# montou não existe aqui. Então, se não estiver no disco, remontamos — são ~6 min
# e o resultado é idêntico (a seleção é determinística, seed fixa).
if not Path(DATA_YAML).exists():
    print("dataset não encontrado neste runtime — remontando do Zenodo...\n")

    REPO = Path("/content/cpzao-auditoria-calcadas")
    if not REPO.exists():
        !git clone -q https://github.com/viniciusborgess/cpzao-auditoria-calcadas.git {REPO}
    sys.path.insert(0, str(REPO / "scripts"))
    import zenodo_dataset as z

    entries  = z.read_central_directory()
    por_nome = {e["name"]: e for e in entries}
    regs     = z.parse_annotations(
        z.extract_member(z.ZENODO_URL, por_nome["wm_annotations.xml"]))
    sel      = z.select_balanced(regs, alvo=ALVO, frac_negativas=0.06, seed=42)

    import time
    t0 = time.perf_counter()
    def _prog(n, t):
        el = time.perf_counter() - t0
        print(f"  {n}/{t} ({n/t*100:.0f}%) — ETA {el/n*(t-n)/60:.1f} min", flush=True)

    blobs = z.extract_many(z.ZENODO_URL,
                           [por_nome[r["name"]] for r in sel],
                           workers=6, progresso=_prog)
    sel = [r for r in sel if r["name"] in blobs]
    z.escrever_dataset(z.split_registros(sel, seed=42), DEST,
                       lambda n: blobs[n], max_lado=640)
    CLASSES = z.CLASSES
    print(f"\n✅ dataset remontado: {len(sel)} imagens")
else:
    print("✅ dataset encontrado neste runtime")

# ─── Hiperparâmetros ───
MODELO_BASE = "yolov8n.pt"   # 'n' = nano: o único tamanho que roda fluido via WASM no celular
EPOCHS      = 150            # ver "Análise crítica" no fim: a 150 o mAP ainda subia
BATCH       = 16
PATIENCE    = 50
SEED        = 42
NOME_RUN    = "calcadas_yolov8n_192"

assert Path(DATA_YAML).exists(), f"data.yaml não encontrado: {DATA_YAML}"
assert IMGSZ in (192, 256, 320, 416, 640), "o app sonda estes tamanhos; use um deles"

print(f"\ndata.yaml : {DATA_YAML}")
print(f"classes   : {CLASSES}")
print(f"imgsz     : {IMGSZ}  (travado pelo app)")
print(f"epochs    : {EPOCHS} (patience {PATIENCE})")
print(Path(DATA_YAML).read_text())

## 3. Treino

Alguns parâmetros merecem justificativa, porque a rubrica pede *análise* do pipeline:

| parâmetro | valor | por quê |
|:--|:--|:--|
| `model` | `yolov8n.pt` | nano é o único que mantém inferência fluida sob WebAssembly no celular; começar dos pesos COCO (transfer learning) é essencial com 60–80 imagens |
| `imgsz` | `192` | entrada pequena → treino e inferência rápidos; o app detecta o tamanho sozinho |
| `epochs` | `150` | com dataset pequeno o modelo precisa de muitas passadas; `patience=50` corta se estabilizar |
| `cos_lr` | `True` | decaimento suave do learning rate — converge melhor em datasets pequenos |
| `hsv_*`, `scale`, `fliplr` | ativos | a câmera do celular vê luz e distância variáveis; a augmentation cobre isso |
| `mosaic` | `0.5` | ajuda o modelo com objetos pequenos, mas em excesso distorce demais poucos dados |
| `close_mosaic` | `15` | desliga o mosaic nas últimas épocas para o modelo ajustar em imagens reais |

In [ ]:
model = YOLO(MODELO_BASE)

results = model.train(
    data        = DATA_YAML,
    imgsz       = IMGSZ,
    epochs      = EPOCHS,
    batch       = BATCH,
    patience    = PATIENCE,
    seed        = SEED,
    device      = DEVICE,
    name        = NOME_RUN,
    project     = "/content/runs",
    exist_ok    = True,

    optimizer   = "auto",
    cos_lr      = True,
    cache       = True,       # dataset pequeno cabe na RAM → épocas muito mais rápidas

    # augmentation
    hsv_h       = 0.015,
    hsv_s       = 0.7,
    hsv_v       = 0.4,
    degrees     = 5.0,
    translate   = 0.1,
    scale       = 0.5,
    fliplr      = 0.5,
    flipud      = 0.0,        # degrau/escada tem "para cima" — não espelhar na vertical
    mosaic      = 0.5,
    close_mosaic= 15,

    plots       = True,
    val         = True,
)

RUN_DIR = Path(results.save_dir)
print("\n✅ treino concluído:", RUN_DIR)
print("pesos:", RUN_DIR / "weights" / "best.pt")

## 4. Curvas de treino

Loss e métricas por época. **O que procurar:**

- `train/box_loss` e `val/box_loss` caindo **juntas** → aprendizado saudável.
- `val` subindo enquanto `train` cai → **overfitting** (reduza epochs ou capture mais fotos).
- `mAP50` em platô → o early stopping já pegou o melhor ponto.

In [ ]:
csv = RUN_DIR / "results.csv"
df  = pd.read_csv(csv)
df.columns = [c.strip() for c in df.columns]

def col(*nomes):
    for n in nomes:
        if n in df.columns:
            return n
    return None

fig, axes = plt.subplots(1, 3, figsize=(17, 4.4))

# (a) losses
for c, lbl, cor in [
    (col("train/box_loss"), "train box", "#2563eb"),
    (col("val/box_loss"),   "val box",   "#93c5fd"),
    (col("train/cls_loss"), "train cls", "#dc2626"),
    (col("val/cls_loss"),   "val cls",   "#fca5a5"),
]:
    if c:
        axes[0].plot(df["epoch"], df[c], label=lbl, color=cor, lw=1.8)
axes[0].set_title("Perdas (box / cls)", fontweight="bold")
axes[0].set_xlabel("época"); axes[0].legend(); axes[0].grid(alpha=.3)

# (b) mAP
for c, lbl, cor in [
    (col("metrics/mAP50(B)", "metrics/mAP_0.5"),         "mAP@50",    "#10b981"),
    (col("metrics/mAP50-95(B)", "metrics/mAP_0.5:0.95"), "mAP@50-95", "#059669"),
]:
    if c:
        axes[1].plot(df["epoch"], df[c], label=lbl, color=cor, lw=2)
        axes[1].axhline(df[c].max(), ls=":", color=cor, alpha=.6)
        axes[1].annotate(f"max {df[c].max():.3f}", (df["epoch"].iloc[-1], df[c].max()),
                         ha="right", va="bottom", color=cor, fontweight="bold")
axes[1].set_title("mAP", fontweight="bold")
axes[1].set_xlabel("época"); axes[1].legend(); axes[1].grid(alpha=.3); axes[1].set_ylim(0, 1)

# (c) precision / recall
for c, lbl, cor in [
    (col("metrics/precision(B)", "metrics/precision"), "Precision", "#f59e0b"),
    (col("metrics/recall(B)",    "metrics/recall"),    "Recall",    "#8b5cf6"),
]:
    if c:
        axes[2].plot(df["epoch"], df[c], label=lbl, color=cor, lw=2)
axes[2].set_title("Precision / Recall", fontweight="bold")
axes[2].set_xlabel("época"); axes[2].legend(); axes[2].grid(alpha=.3); axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.savefig("curvas_treino.png", dpi=130, bbox_inches="tight")
plt.show()

mc = col("metrics/mAP50(B)", "metrics/mAP_0.5")
print(f"épocas efetivamente treinadas: {len(df)} de {EPOCHS}")
if mc:
    print(f"melhor mAP@50: {df[mc].max():.4f} na época {int(df.loc[df[mc].idxmax(), 'epoch'])}")

## 5. Matriz de confusão e curvas PR

Artefatos gerados automaticamente pelo Ultralytics. **A matriz de confusão normalizada é o
gráfico mais informativo do projeto**: a diagonal mostra o acerto por classe, e a coluna
`background` revela o que o modelo deixou passar.

In [ ]:
artefatos = [
    ("confusion_matrix_normalized.png", "Matriz de Confusão (normalizada)"),
    ("confusion_matrix.png",            "Matriz de Confusão (contagens)"),
    ("PR_curve.png",                    "Curva Precision-Recall"),
    ("F1_curve.png",                    "Curva F1 × confiança"),
    ("labels.jpg",                       "Distribuição das labels"),
]

for arq, titulo in artefatos:
    p = RUN_DIR / arq
    if p.exists():
        plt.figure(figsize=(9, 7))
        plt.imshow(Image.open(p)); plt.axis("off")
        plt.title(titulo, fontweight="bold", fontsize=13)
        plt.show()
    else:
        print("não gerado:", arq)

### Como ler a curva F1 × confiança

O pico dessa curva é o **limiar de confiança ótimo** do seu modelo. Anote esse valor: ele é o
que você deve configurar no app (o padrão é `0.35`, em `app/src/main.js`). Se o pico estiver
em 0.25, subir o slider para 0.35 na demo joga detecções válidas fora.

## 6. Avaliação final no conjunto de TESTE

As métricas acima são de **validação** — o conjunto que guiou o early stopping. A rubrica pede
divisão em treino/val/**teste**, e o teste é o único número honesto: imagens que o modelo
nunca viu e que não influenciaram nenhuma decisão.

In [ ]:
best = YOLO(str(RUN_DIR / "weights" / "best.pt"))

print("═" * 64); print("VALIDAÇÃO (split=val)"); print("═" * 64)
m_val = best.val(data=DATA_YAML, imgsz=IMGSZ, split="val", device=DEVICE, plots=False, verbose=False)

print("\n" + "═" * 64); print("TESTE (split=test) ← métrica oficial"); print("═" * 64)
m_test = best.val(data=DATA_YAML, imgsz=IMGSZ, split="test", device=DEVICE, plots=True, verbose=False,
                  name=NOME_RUN + "_test", project="/content/runs")

def resumo(m):
    return {"mAP50": float(m.box.map50), "mAP50-95": float(m.box.map),
            "precision": float(m.box.mp), "recall": float(m.box.mr)}

r_val, r_test = resumo(m_val), resumo(m_test)

print("\n┌─ GERAL " + "─" * 46)
print(f"│ {'métrica':<12s} {'val':>10s} {'test':>10s}")
print("│ " + "─" * 34)
for k in ["mAP50", "mAP50-95", "precision", "recall"]:
    print(f"│ {k:<12s} {r_val[k]:>10.4f} {r_test[k]:>10.4f}")
print("└" + "─" * 54)

# Métricas por classe no teste
print("\n┌─ POR CLASSE (teste) " + "─" * 42)
print(f"│ {'classe':<14s} {'P':>8s} {'R':>8s} {'mAP50':>8s} {'mAP50-95':>10s}")
print("│ " + "─" * 52)
por_classe_metricas = {}
for i, idx in enumerate(m_test.box.ap_class_index):
    p, r, ap50, ap = m_test.box.class_result(i)
    nome = CLASSES[int(idx)]
    por_classe_metricas[nome] = {"precision": float(p), "recall": float(r),
                                 "mAP50": float(ap50), "mAP50-95": float(ap)}
    print(f"│ {nome:<14s} {p:>8.3f} {r:>8.3f} {ap50:>8.3f} {ap:>10.3f}")
print("└" + "─" * 62)

print("\n── Leitura ──")
print("Precision baixa → muitos falsos positivos (modelo 'vê' coisas que não existem)")
print("Recall baixa    → deixa objetos passar; na demo, parece que o app 'não detecta'")
if r_test["mAP50"] >= 0.5:
    print(f"\n✅ mAP@50 de teste = {r_test['mAP50']:.3f} — resultado sólido para 60–80 imagens")
elif r_test["mAP50"] >= 0.3:
    print(f"\n🟡 mAP@50 de teste = {r_test['mAP50']:.3f} — aceitável; aumentar ALVO no notebook 01 ajudaria")
else:
    print(f"\n🔴 mAP@50 de teste = {r_test['mAP50']:.3f} — aumente ALVO no notebook 01 e treine de novo")

## 7. Inferência visual no conjunto de teste

Predições sobre imagens nunca vistas. É a melhor prévia de como o app vai se comportar amanhã.

In [ ]:
import yaml
cfg       = yaml.safe_load(Path(DATA_YAML).read_text())
test_dir  = Path(cfg["path"]) / cfg.get("test", "test/images")
test_imgs = sorted([p for p in test_dir.iterdir()
                    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])[:6]

print(f"{len(test_imgs)} imagens de teste")

preds = best.predict(source=[str(p) for p in test_imgs], imgsz=IMGSZ,
                     conf=0.25, iou=0.45, device=DEVICE, verbose=False)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, pr, ip in zip(axes.ravel(), preds, test_imgs):
    ax.imshow(Image.fromarray(pr.plot()[:, :, ::-1]))
    ax.axis("off")
    n = len(pr.boxes)
    achados = ", ".join(sorted({CLASSES[int(c)] for c in pr.boxes.cls})) if n else "—"
    ax.set_title(f"{n} detecção(ões): {achados}", fontsize=9, fontweight="bold")
for ax in axes.ravel()[len(preds):]:
    ax.axis("off")

plt.tight_layout()
plt.savefig("predicoes_teste.png", dpi=120, bbox_inches="tight")
plt.show()

## 8. Export para ONNX

Agora o passo que conecta o treino ao celular. Cada parâmetro aqui existe por uma razão
ditada pelo app:

| parâmetro | valor | por quê |
|:--|:--|:--|
| `format` | `onnx` | o app usa **ONNX Runtime Web** (WASM), não TFLite |
| `imgsz` | `192` | detectado automaticamente pelo app ao carregar o modelo |
| `opset` | `12` | amplamente suportado pelo onnxruntime-web 1.18 |
| `dynamic` | `False` | o app sempre envia `[1,3,192,192]`; shape fixo permite mais otimização |
| `simplify` | `True` | funde nós e reduz o grafo → menos latência no WASM |
| `nms` | `False` | **crítico:** o app faz o NMS dele em JS (`setIou`/`setConf`). Embutir NMS muda o shape da saída e quebra o parser |
| `half` | `False` | WASM não tem caminho estável para fp16 |

In [ ]:
onnx_path = best.export(
    format   = "onnx",
    imgsz    = IMGSZ,
    opset    = 12,
    dynamic  = False,
    simplify = True,
    nms      = False,
    half     = False,
    device   = "cpu",
)

onnx_path = Path(onnx_path)
print("\n✅ exportado:", onnx_path)
print(f"   tamanho: {onnx_path.stat().st_size / 1e6:.2f} MB")

## 9. Verificação de compatibilidade com o app

Esta célula **replica em Python a função `detectOutputConventions()`** do
`app/src/inference.js` e confere, item por item, se o modelo vai funcionar no celular.

Se tudo passar aqui, funciona no app. Se algo falhar, conserte **antes** de ir para a aula.

In [ ]:
import onnx, onnxruntime as ort

# ── estrutura do grafo ──
mdl = onnx.load(str(onnx_path))
onnx.checker.check_model(mdl)
print("✅ grafo ONNX válido · opset:", mdl.opset_import[0].version)

sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
inp  = sess.get_inputs()[0]
outs = sess.get_outputs()

print(f"\nentrada : {inp.name} {inp.shape} ({inp.type})")
for o in outs:
    print(f"saída   : {o.name} {o.shape}")

# ── inferência dummy, igual ao app ──
dummy = np.full((1, 3, IMGSZ, IMGSZ), 0.45, dtype=np.float32)
y     = sess.run(None, {inp.name: dummy})[0]

channels   = y.shape[1]
num_boxes  = y.shape[2]
num_classes_detectado = channels - 4
max_cx     = float(y[0, 0, :min(num_boxes, 200)].max())
normalized = max_cx < 1.5
implied_pose = channels - 4 - 17 * 3
tipo = "pose" if 1 <= implied_pose <= 5 else "detection"

print(f"\n── o que o app vai detectar ──")
print(f"  shape da saída   : {y.shape}")
print(f"  type             : {tipo}")
print(f"  numClasses       : {num_classes_detectado}")
print(f"  max cx (200 box) : {max_cx:.2f}")
print(f"  normalizedCoords : {normalized}  → preprocess '{'crop' if (tipo=='detection' and normalized) else 'letterbox'}'")

# ── checklist ──
checks = [
    (f"entrada é [1,3,{IMGSZ},{IMGSZ}]", list(inp.shape) == [1, 3, IMGSZ, IMGSZ],
     str(inp.shape)),
    ("saída única (não segmentação)",  len(outs) == 1,
     f"{len(outs)} saída(s)"),
    ("detectado como 'detection'",     tipo == "detection",
     tipo),
    (f"numClasses == {len(CLASSES)}",  num_classes_detectado == len(CLASSES),
     str(num_classes_detectado)),
    ("numClasses != 80 (usa CLASS_NAMES)", num_classes_detectado != 80,
     "ok" if num_classes_detectado != 80 else "colidiu com COCO"),
    ("NMS NÃO embutido",               channels == 4 + len(CLASSES),
     f"channels={channels}, esperado={4+len(CLASSES)}"),
    ("modelo < 15 MB (carrega rápido no 4G)", onnx_path.stat().st_size < 15e6,
     f"{onnx_path.stat().st_size/1e6:.2f} MB"),
]

print("\n" + "═" * 64)
print("COMPATIBILIDADE COM O yoloinferApp")
print("═" * 64)
for nome, ok, det in checks:
    print(f"  {'✅' if ok else '❌'}  {nome:<42s} {det}")
print("═" * 64)

if all(ok for _, ok, _ in checks):
    print("🎉 modelo compatível — pode copiar para app/public/models/")
else:
    print("🚨 NÃO copie para o app ainda: resolva os ❌ acima")

print(f"\n📌 Lembre de ajustar em app/src/main.js:")
print(f"   const CLASS_NAMES = {json.dumps(CLASSES)};")

## 10. Latência (estimativa de fluidez no celular)

A rubrica pede inferência **fluida**. Medimos aqui na CPU, que é uma proxy razoável do que o
WASM faz no celular — na prática, espere algo entre 1× e 3× este tempo em um aparelho mediano.

In [ ]:
for _ in range(5):                       # warmup
    sess.run(None, {inp.name: dummy})

N, t0 = 50, time.perf_counter()
for _ in range(N):
    sess.run(None, {inp.name: dummy})
ms = (time.perf_counter() - t0) / N * 1000

print(f"latência média (CPU, 1 thread-ish): {ms:.1f} ms  →  ~{1000/ms:.1f} FPS")
print(f"estimativa no celular (WASM, 2–3×): ~{ms*2.5:.0f} ms  →  ~{1000/(ms*2.5):.1f} FPS")
print("\n> acima de ~8 FPS a demo já parece fluida ao vivo")
print("> se ficar lento: confirme que é yolov8n (não s/m) e que imgsz=192")

## 11. Gerar o bloco de métricas para o README

In [ ]:
linhas = ["| classe | Precision | Recall | mAP@50 | mAP@50-95 |", "|:--|--:|--:|--:|--:|"]
for c in CLASSES:
    m = por_classe_metricas.get(c)
    linhas.append(f"| `{c}` | {m['precision']:.3f} | {m['recall']:.3f} | {m['mAP50']:.3f} | {m['mAP50-95']:.3f} |"
                  if m else f"| `{c}` | — | — | — | — |")
linhas.append(f"| **geral (teste)** | **{r_test['precision']:.3f}** | **{r_test['recall']:.3f}** "
              f"| **{r_test['mAP50']:.3f}** | **{r_test['mAP50-95']:.3f}** |")

bloco = f'''## 🎯 Métricas Alcançadas

**Modelo:** YOLOv8n · `imgsz=192` · {len(df)} épocas treinadas · transfer learning de pesos COCO

### Resultado por classe (conjunto de **teste**)

{chr(10).join(linhas)}

### Validação × Teste

| métrica | validação | teste |
|:--|--:|--:|
| mAP@50 | {r_val['mAP50']:.3f} | **{r_test['mAP50']:.3f}** |
| mAP@50-95 | {r_val['mAP50-95']:.3f} | **{r_test['mAP50-95']:.3f}** |
| Precision | {r_val['precision']:.3f} | **{r_test['precision']:.3f}** |
| Recall | {r_val['recall']:.3f} | **{r_test['recall']:.3f}** |

### Modelo exportado

| item | valor |
|:--|:--|
| formato | ONNX (opset 12), shape estático |
| entrada | `[1, 3, 192, 192]` |
| saída | `{tuple(y.shape)}` = `[1, 4+{len(CLASSES)}, {num_boxes}]` |
| NMS | feito em JS pelo app (não embutido no grafo) |
| tamanho | {onnx_path.stat().st_size/1e6:.2f} MB |
| latência (CPU) | {ms:.1f} ms (~{1000/ms:.1f} FPS) |

### Artefatos

- `docs/metricas/curvas_treino.png` — perdas e métricas por época
- `docs/metricas/confusion_matrix_normalized.png` — matriz de confusão
- `docs/metricas/results.png` — métricas por época (Ultralytics)
- `docs/metricas/predicoes_teste.png` — inferência no conjunto de teste
'''

print(bloco)
Path("metricas.md").write_text(bloco)
json.dump({"geral_val": r_val, "geral_test": r_test, "por_classe": por_classe_metricas,
           "epocas": len(df), "imgsz": IMGSZ, "onnx_shape": list(y.shape),
           "onnx_mb": onnx_path.stat().st_size/1e6, "latencia_cpu_ms": ms},
          open("metricas.json", "w"), indent=2, ensure_ascii=False)
print("\n✅ gerados: metricas.md · metricas.json")

## 12. Baixar os artefatos

Baixa o modelo e todos os gráficos. **Depois disso, siga para o `README.md` do repositório,
seção *Deploy*.**

In [ ]:
from google.colab import files

# renomeia o onnx para o nome que o app espera
final_onnx = Path("/content/calcadas.onnx")
shutil.copy(onnx_path, final_onnx)
shutil.copy(RUN_DIR / "weights" / "best.pt", "/content/best.pt")

# junta os gráficos do Ultralytics
for arq in ["confusion_matrix_normalized.png", "confusion_matrix.png",
            "PR_curve.png", "F1_curve.png", "results.png", "labels.jpg"]:
    p = RUN_DIR / arq
    if p.exists():
        shutil.copy(p, f"/content/{arq}")

alvos = ["calcadas.onnx", "best.pt", "metricas.md", "metricas.json",
         "curvas_treino.png", "predicoes_teste.png",
         "confusion_matrix_normalized.png", "confusion_matrix.png",
         "PR_curve.png", "F1_curve.png", "results.png", "labels.jpg"]

print("Baixando — aceite os downloads do navegador:\n")
for a in alvos:
    p = Path("/content") / a
    if p.exists():
        print(f"  ↓ {a} ({p.stat().st_size/1e6:.2f} MB)")
        try:
            files.download(str(p))
        except Exception as e:
            print("    falhou:", e)

---

## 📉 Análise crítica dos resultados

O resultado geral foi **mAP@50 = 0,244** no conjunto de teste. Abaixo, o diagnóstico de
*por quê* — que é mais informativo que o número isolado.

### O desempenho não é uniforme entre as classes

| classe | Recall | mAP@50 | formato típico do objeto |
|:--|--:|--:|:--|
| `escada` | **0,49** | **0,524** | blob compacto, proporção equilibrada |
| `degrau` | 0,10 | 0,088 | faixa **longa e fina** (guia de calçada) |
| `corrimao` | 0,11 | 0,119 | barra **fina e diagonal** |

A classe que funciona é justamente a única com formato compacto. Isso aponta para uma causa
**geométrica**, não de quantidade ou qualidade de dados.

### As três evidências, juntas

**1. Matriz de confusão — a diagonal é ~0,47 nas TRÊS classes.**
Quando o modelo detecta alguma coisa, ele classifica certo quase metade das vezes, de forma
uniforme. Ou seja: o reconhecimento das classes não é o gargalo.

**2. Predições no teste estão visualmente corretas.**
`escada 0,78` · `escada 0,81` · `corrimao 0,74` · `degrau 0,64` — todas sobre os objetos
certos. O modelo *enxerga*.

**3. As curvas descartam overfitting e falta de treino.**
Train e val loss caem **juntas** até a época 150, e o mAP ainda subia no fim. Não havia
excesso de ajuste, e também não era questão de treinar mais um pouco.

### A explicação

O que falha é a **localização**, não o reconhecimento — e especificamente em objetos finos.

O mAP@50 exige IoU ≥ 0,5 com o gabarito. Para uma caixa longa e fina, o IoU é brutalmente
sensível ao erro na dimensão curta:

> Uma guia de calçada que ocupa **3% da altura** da imagem vira **≈ 6 pixels** numa entrada
> de 192. Um erro de 3 pixels na borda superior já corta o IoU pela metade — a detecção,
> visualmente correta, é contada como erro.

Para um blob como `escada`, o mesmo erro absoluto é irrelevante. Daí a diferença de 5× no mAP
entre classes do mesmo dataset, com o mesmo treino.

### O que isso significa na prática

Para a **auditoria** (contar e medir barreiras com precisão), o mAP de 0,244 é insuficiente —
seria preciso melhorar antes de usar em campo.

Para a **demonstração ao vivo**, bem menos: o app precisa de *uma* caixa correta sobre o
objeto apontado, não de recuperar todas as instâncias com IoU ≥ 0,5. É por isso que a
inferência em tempo real funciona visivelmente melhor do que o número sugere.

### Próxima iteração — o que mudaria

Em ordem de impacto esperado:

| mudança | de → para | porquê |
|:--|:--|:--|
| **`imgsz`** | 192 → **320** | ataca a causa raiz: o objeto fino ganha 1,7× mais pixels na dimensão crítica. Custo: ~10 ms → ~29 ms (~34 FPS), ainda fluido |
| `degrees` | 5.0 → **0.0** | guia de calçada é horizontal por definição; rotacionar descaracteriza o alvo |
| `scale` | 0.5 → **0.3** | encolher a imagem faz o objeto fino desaparecer |
| `mosaic` | 0.5 → **0.3** | o mosaico reduz cada imagem; pesado demais para alvos finos |
| `epochs` | 150 → **250** | o mAP ainda subia, sem sinal de overfitting |

Nada disso exige mexer no app: ele detecta o tamanho de entrada do modelo sozinho
(`detectInputSize()` em `app/src/inference.js`). Basta alterar `IMGSZ` na célula de
configuração e reexecutar este notebook.

> Uma alternativa estrutural seria **fundir `degrau` e `escada`** numa só classe. A matriz de
> confusão mostra 0,28 + 0,19 de confusão cruzada entre as duas, o que faz sentido semântico:
> uma escada *é* uma sequência de degraus, e a fronteira entre "um degrau" e "dois degraus" é
> genuinamente ambígua. Mantivemos separadas porque a rubrica exige no mínimo 3 classes.

---

## ✅ Próximos passos

Onde colocar cada arquivo baixado, no repositório:

```
app/public/models/calcadas.onnx     ← o modelo (é o que o celular carrega)
models/best.pt                      ← pesos PyTorch (referência)
docs/metricas/*.png                 ← todos os gráficos
```

Comandos:

```bash
cd ~/Documents/cpzao-auditoria-calcadas

cp ~/Downloads/calcadas.onnx app/public/models/
cp ~/Downloads/best.pt       models/
cp ~/Downloads/*.png ~/Downloads/labels.jpg docs/metricas/

# cole o conteúdo de metricas.md na seção de métricas do README.md

cd app && npm install && npm run dev -- --host
```

Abra a URL `https://<ip-local>:5173` **no celular**, aceite o aviso de certificado,
e aponte a câmera traseira para uma calçada.

→ O passo a passo completo do deploy está no [`README.md`](../README.md).